# 03b — Phân lớp Logistic Regression trên validation

Lưới đặt trước: base/physical × C 0.01/0.1/1/10 × class_weight none/balanced = 16 cấu hình mỗi giao thức. Fit transformer và classifier chỉ trên train, chọn bằng AP validation, ngưỡng max F1 validation. Test chưa được nạp. Xem docs/CLASSIFICATION.md để biết giả định và giới hạn so sánh.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve()
while not (ROOT / "configs/classification.json").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Mở notebook trong repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from maintenance.classification import run_classification
manifest = run_classification(ROOT)
assert manifest["test_evaluated"] is False
print("Đã chạy Logistic Regression, chưa mở test.")

Đã chạy Logistic Regression, chưa mở test.


In [2]:
candidates = pd.read_csv(ROOT / "reports/tables/classification/candidates.csv")
display(candidates[["split_id", "candidate_id", "n_iter", "average_precision", "precision_at_50"]])
comparison = pd.read_csv(ROOT / "reports/tables/classification/comparison.csv")
display(comparison[["split_id", "model_family", "average_precision", "roc_auc", "tp_at_50", "precision_at_50", "recall_at_50", "tp", "fp", "fn", "tn", "precision", "recall", "f1"]])

,split_id,candidate_id,n_iter,average_precision,precision_at_50
0,ordered-gap100-v1,lr-base-C0.01-none,11,0.501111,0.50
1,ordered-gap100-v1,lr-base-C0.01-balanced,17,0.466271,0.44
2,ordered-gap100-v1,lr-base-C0.1-none,20,0.462773,0.44
3,ordered-gap100-v1,lr-base-C0.1-balanced,23,0.385465,0.38
4,ordered-gap100-v1,lr-base-C1-none,26,0.398434,0.40
5,ordered-gap100-v1,lr-base-C1-balanced,19,0.353500,0.36
6,ordered-gap100-v1,lr-base-C10-none,20,0.386782,0.38
7,ordered-gap100-v1,lr-base-C10-balanced,19,0.351573,0.36
8,ordered-gap100-v1,lr-physical-C0.01-none,12,0.383595,0.36
9,ordered-gap100-v1,lr-physical-C0.01-balanced,22,0.361576,0.32


,split_id,model_family,average_precision,roc_auc,tp_at_50,precision_at_50,recall_at_50,tp,fp,fn,tn,precision,recall,f1
0,ordered-gap100-v1,isolation_forest,0.320247,0.859300,15.0,0.30,0.348837,15,31,28,1826,0.326087,0.348837,0.337079
1,ordered-gap100-v1,random,0.019587,0.454835,0.0,0.00,0.000000,43,1747,0,110,0.024022,1.000000,0.046918
2,ordered-gap100-v1,always_normal,0.022632,0.500000,NaN,NaN,NaN,0,0,43,1857,NaN,0.000000,0.000000
3,stratified-seed42-v1,isolation_forest,0.258498,0.842346,20.0,0.40,0.294118,20,25,48,1907,0.444444,0.294118,0.353982
4,stratified-seed42-v1,random,0.039845,0.575676,0.0,0.00,0.000000,28,485,40,1447,0.054581,0.411765,0.096386
5,stratified-seed42-v1,always_normal,0.034000,0.500000,NaN,NaN,NaN,0,0,68,1932,NaN,0.000000,0.000000
6,ordered-gap100-v1,logistic_regression,0.501111,0.829708,25.0,0.50,0.581395,24,13,19,1844,0.648649,0.558140,0.600000
7,stratified-seed42-v1,logistic_regression,0.463414,0.894920,26.0,0.52,0.382353,27,24,41,1908,0.529412,0.397059,0.453782


## Diễn giải

Logistic Regression học bằng X_train và y_train, khác Isolation Forest fit không dùng y. Tuy nhiên cả hai đã dùng nhãn validation để chọn cấu hình/ngưỡng. Kết quả validation này có thiên lệch lựa chọn; không coi chênh lệch là tác động thuần của việc thêm nhãn vì thuật toán/lưới khác nhau.

ROC-AUC, AP và top-k đo các khía cạnh khác nhau. Không đồng nhất điểm xác suất lớp 1 với xác suất hỏng đã được hiệu chuẩn ngoài nhà máy, đặc biệt khi thử class_weight=balanced. Không suy hệ số thành nguyên nhân gây hỏng; các biến có tương quan và Type one-hot đầy đủ.

In [3]:
for selection in manifest["selections"]:
    print(selection["split_id"], selection["candidate_id"], "threshold=", selection["threshold"])
print("Cấu hình/ngưỡng đã lưu theo validation; bước 8 mới đánh giá cuối.")

ordered-gap100-v1 lr-base-C0.01-none threshold= 0.14391614297042812
stratified-seed42-v1 lr-physical-C1-none threshold= 0.3030976135259637
Cấu hình/ngưỡng đã lưu theo validation; bước 8 mới đánh giá cuối.
